# PDF to Audiobook — Chatterbox TTS Voice Cloning
Converts a book into chapter-by-chapter MP3s using a cloned voice.

**Before running:**
1. Make sure GPU is enabled: `Runtime → Change runtime type → T4 GPU`
2. Upload your chapter `.txt` files and voice reference to Google Drive
3. Mount Drive (Cell 3)
4. **Edit Cell 4 — set ALL your paths there before running anything else**

**Need a test book? Free public domain plain text:**
- [Project Gutenberg](https://www.gutenberg.org) — thousands of classic books
- [The Adventures of Sherlock Holmes](https://www.gutenberg.org/ebooks/1661) — short stories, clear chapters
- [The Time Machine – H.G. Wells](https://www.gutenberg.org/ebooks/35) — short, fast to test

**Workflow for a new book:**
```
1. Download plain text (.txt) from Gutenberg → upload to Drive
2. Run Cell A (below) to clean text and split into chapters
3. Set TXT_DIR in Cell 4 → run Cells 5 → 6 → 7 → 8 → 9
```

In [ ]:
# ── Cell 0: Pull latest code from GitHub ─────────────────────────
# Run this once at the start of every session.
# Opens the repo so all scripts are available in this runtime.

import os

REPO_URL = "https://github.com/Mashfique/Audio_Booker.git"
REPO_DIR = "/content/Audio_Booker"

if os.path.exists(REPO_DIR):
    # Already cloned — just pull latest changes
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print(f"\nRepo ready at: {REPO_DIR}")
print("Files available:")
!ls

In [ ]:
# ── Cell 1: Check GPU + Python version ───────────────────────────
import sys, torch

print(f"Python : {sys.version}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU    : {gpu.name}")
    print(f"VRAM   : {gpu.total_memory / 1e9:.1f} GB")
else:
    print("No GPU — go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# ── Cell 2: Install packages ──────────────────────────────────────
%pip install -q chatterbox-tts pdfplumber

# Fix torchvision to match torch 2.6.0 that chatterbox-tts installs
%pip install -q torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 --force-reinstall

# Numba (used by librosa, used by chatterbox) needs numpy <= 2.0
%pip install -q "numpy==1.26.4" --force-reinstall

# !! After this finishes → Runtime → Restart session → re-run this cell → continue from Cell 3

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted. Use the folder icon in the left sidebar to browse files.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ── CELL 4: ALL CONFIGURATION — edit this cell before running anything else ──
# ══════════════════════════════════════════════════════════════════════════════
import os
from pathlib import Path

# ── Google Drive base folder ──────────────────────────────────────────────────
# Change this if your audiobook folder has a different name on Drive
DRIVE_BASE = "/content/drive/MyDrive/audiobook"

# ── Input ─────────────────────────────────────────────────────────────────────
TXT_DIR   = f"{DRIVE_BASE}/chapters"       # folder of per-chapter .txt files
VOICE_REF = f"{DRIVE_BASE}/reference.mp3"  # your voice cloning reference clip

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR      = f"{DRIVE_BASE}/output"   # per-chapter MP3s are saved here
FULL_OUTPUT_DIR = f"{DRIVE_BASE}"          # stitched full audiobook saved here

# ── Model cache ───────────────────────────────────────────────────────────────
# Persists the ~1.5 GB model weights on Drive — avoids re-downloading each session
HF_CACHE_DIR = f"{DRIVE_BASE}/hf_cache"

# ── TTS settings ──────────────────────────────────────────────────────────────
REF_SECONDS = 15    # seconds to use from reference audio (6–30 ideal)
CHUNK_WORDS = 200   # words per TTS chunk (keep under 250)

# ── Derived paths — do not edit ───────────────────────────────────────────────
REF_WAV = "/content/reference_trimmed.wav"   # trimmed reference, lives in Colab runtime only

# ── Create all output directories ─────────────────────────────────────────────
for _d in [TXT_DIR, OUTPUT_DIR, FULL_OUTPUT_DIR, HF_CACHE_DIR]:
    Path(_d).mkdir(parents=True, exist_ok=True)

# ── Print summary ─────────────────────────────────────────────────────────────
W = 60
print("═" * W)
print("  CONFIGURATION SUMMARY")
print("═" * W)
print(f"  Drive base       : {DRIVE_BASE}")
print(f"  Chapters (input) : {TXT_DIR}")
print(f"  Voice reference  : {VOICE_REF}")
print(f"  Chapter MP3s out : {OUTPUT_DIR}")
print(f"  Full audiobook   : {FULL_OUTPUT_DIR}")
print(f"  Model cache      : {HF_CACHE_DIR}")
print(f"  Ref WAV (temp)   : {REF_WAV}")
print("─" * W)
print(f"  Reference clip   : {REF_SECONDS}s  |  Chunk size: {CHUNK_WORDS} words")
print("─" * W)
print("  Folders created  : OK (all output directories exist)")
print("═" * W)

In [ ]:
# ── Cell 4A: Prepare a new book (run once per book) ──────────────
# Skip this cell if you already have per-chapter .txt files in TXT_DIR.
#
# 1. Upload your raw .txt file (e.g. from Project Gutenberg) to Drive
# 2. Set RAW_TXT below, then run this cell
# 3. It cleans the text and splits it into numbered chapter files in TXT_DIR

RAW_TXT = f"{DRIVE_BASE}/pg1661.txt"    # ← path to your raw downloaded .txt file

# ── Run tts_cleaner.py ────────────────────────────────────────────
CLEANED_TXT = f"{DRIVE_BASE}/cleaned.txt"
print("Step 1 — Cleaning text...")
!python /content/Audio_Booker/tts_cleaner.py "{RAW_TXT}" "{CLEANED_TXT}"

# ── Run chapter_splitter.py ───────────────────────────────────────
print("\nStep 2 — Splitting into chapters...")
!python /content/Audio_Booker/chapter_splitter.py "{CLEANED_TXT}" "{TXT_DIR}"

print(f"\nDone. Chapter .txt files are in: {TXT_DIR}")
print("You can now run Cells 5 → 6 → 7 → 8 → 9 to start converting.")

In [ ]:
# ── Cell 5: Helper functions ──────────────────────────────────────
import re, subprocess, tempfile, shutil
from pathlib import Path
from collections import Counter
import pdfplumber

CHAPTER_PATTERNS = [
    r"^chapter\s+[\divxlcdm]+", r"^chapter\s+\w+",
    r"^part\s+[\divxlcdm]+",    r"^part\s+\w+",
    r"^\d+\.\s+[A-Z]",
    r"^epilogue$", r"^prologue$", r"^introduction$",
    r"^preface$",  r"^foreword$", r"^appendix", r"^conclusion$",
]

_KNOWN_ACRONYMS = {
    "AI", "ML", "UK", "US", "EU", "UN", "NATO", "FBI", "CIA", "NASA",
    "CEO", "CFO", "CTO", "HR", "IT", "PR", "ID", "OK", "TV", "PC",
    "USB", "PDF", "MP3", "TTS", "GPU", "CPU", "RAM", "API", "URL",
    "HTML", "CSS", "SQL", "GMT", "EST", "PST", "BC", "AD", "WWII", "WWI",
    "USA", "UAE", "PTSD", "DNA", "RNA", "VIP", "RSVP",
}

def _fix_allcaps(text):
    def _replace(m):
        w = m.group(0)
        return w if w in _KNOWN_ACRONYMS else w.capitalize()
    return re.sub(r"\b[A-Z]{3,}\b", _replace, text)

def clean_text(text):
    text = re.sub(r"-\n(\w)", r"\1", text)
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    text = _fix_allcaps(text)
    return text.strip()

def is_chapter_heading(line):
    s = line.strip()
    if not s or len(s) > 80: return False
    return any(re.match(p, s, re.IGNORECASE) for p in CHAPTER_PATTERNS)

def find_repeated_lines(pdf, threshold=5):
    counts = Counter()
    for page in pdf.pages:
        seen = set()
        for line in (page.extract_text() or "").splitlines():
            s = line.strip()
            if s and s not in seen:
                counts[s] += 1; seen.add(s)
    return {l for l, n in counts.items() if n >= threshold}

def extract_chapters(pdf_path):
    chapters, current = [], {"title": "Front Matter", "text": ""}
    with pdfplumber.open(pdf_path) as pdf:
        repeated = find_repeated_lines(pdf)
        for page in pdf.pages:
            for line in (page.extract_text() or "").splitlines():
                s = line.strip()
                if s in repeated: continue
                if is_chapter_heading(s):
                    if current["text"].strip(): chapters.append(current)
                    current = {"title": s, "text": ""}
                else:
                    current["text"] += line + "\n"
    if current["text"].strip(): chapters.append(current)
    return chapters

def split_by_words(text, max_words):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, current, count = [], [], 0
    for s in sentences:
        w = len(s.split())
        if count + w > max_words and current:
            chunks.append(" ".join(current)); current, count = [s], w
        else:
            current.append(s); count += w
    if current: chunks.append(" ".join(current))
    return chunks

def ffmpeg_concat(parts, output):
    with tempfile.TemporaryDirectory() as tmp:
        lst = Path(tmp) / "list.txt"
        lst.write_text("\n".join(f"file '{Path(p).resolve()}'" for p in parts))
        subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0",
                        "-i", str(lst), "-c", "copy", str(output)],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

def wav_to_mp3(wav, mp3):
    subprocess.run(["ffmpeg", "-y", "-i", str(wav), "-b:a", "192k", str(mp3)],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    Path(wav).unlink()

print("Helper functions ready.")

In [ ]:
# ── Cell 6: Prepare reference audio ──────────────────────────────
import subprocess

# Trim to REF_SECONDS starting at 3s (skips any intro noise)
subprocess.run(
    ["ffmpeg", "-y", "-i", VOICE_REF,
     "-ss", "3", "-t", str(REF_SECONDS),
     "-ar", "22050", "-ac", "1", REF_WAV],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True
)
print(f"Reference audio trimmed to {REF_SECONDS}s")
print(f"  Source : {VOICE_REF}")
print(f"  Output : {REF_WAV}")

In [ ]:
# ── Cell 7: Load Chatterbox TTS model ────────────────────────────
# Cache model weights to Drive — only downloads once, loads from Drive every session after
import os
os.environ["HF_HOME"] = HF_CACHE_DIR

import torch
from chatterbox.tts import ChatterboxTTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Chatterbox TTS on {device.upper()}...")
print(f"Model cache : {HF_CACHE_DIR}")

model = ChatterboxTTS.from_pretrained(device=device)
print("Model loaded and ready.")

In [ ]:
# ── Cell 8: Load chapters from .txt files ────────────────────────
txt_files = sorted(Path(TXT_DIR).glob("*.txt"))

if not txt_files:
    raise FileNotFoundError(f"No .txt files found in {TXT_DIR}")

chapters = []
for f in txt_files:
    chapters.append({"title": f.stem, "text": f.read_text(encoding="utf-8")})

print(f"Found {len(chapters)} chapter(s):\n")
for i, ch in enumerate(chapters, 1):
    print(f"  {i:02d}. {ch['title']}  ({len(ch['text'].split()):,} words)")

In [ ]:
# ── Cell 9: Convert chapters to MP3 ──────────────────────────────
import numpy as np
import scipy.io.wavfile as wavfile
from tqdm.notebook import tqdm

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# Pre-calculate all chapters and their chunks
all_chapters = []
for i, ch in enumerate(chapters, 1):
    chunks = split_by_words(ch["text"], CHUNK_WORDS)
    safe   = re.sub(r"[^\w\s-]", "", ch["title"])[:50].strip()
    fname  = f"{i:02d}_{safe}.mp3"
    all_chapters.append((i, ch, chunks, fname))

total_chunks = sum(len(chunks) for _, _, chunks, _ in all_chapters)
done_chunks  = sum(len(chunks) for _, _, chunks, fname in all_chapters
                   if (out_dir / fname).exists())
skipped      = sum(1 for _, _, _, fname in all_chapters if (out_dir / fname).exists())

print(f"{len(chapters)} chapters — {total_chunks} chunks total")
if skipped:
    print(f"Resuming: {skipped} chapter(s) already done, skipping them\n")

overall = tqdm(total=total_chunks, initial=done_chunks,
               desc="Overall", unit="chunk", ncols=70)

for i, ch, chunks, fname in all_chapters:
    out = out_dir / fname

    if out.exists():
        print(f"[{i}/{len(chapters)}] SKIP — {ch['title']} (already converted)")
        continue

    overall.set_description(f"Ch {i}/{len(chapters)}")
    print(f"\n[{i}/{len(chapters)}] {ch['title']} — {len(chunks)} chunk(s)")

    with tempfile.TemporaryDirectory() as tmp:
        parts = []
        for j, chunk in enumerate(chunks):
            wav = f"{tmp}/chunk_{j:04d}.wav"
            mp3 = f"{tmp}/chunk_{j:04d}.mp3"

            wav_tensor = model.generate(chunk, audio_prompt_path=ref_wav)
            audio_np = wav_tensor.squeeze().cpu().numpy()
            wavfile.write(wav, model.sr, audio_np)
            wav_to_mp3(wav, mp3)
            parts.append(mp3)
            overall.update(1)

        if len(parts) == 1:
            shutil.copy(parts[0], out)
        else:
            ffmpeg_concat(parts, out)

    size_kb = out.stat().st_size // 1024
    print(f"  → {fname} ({size_kb} KB)")

overall.close()
print(f"\nDone! {len(chapters)} MP3(s) saved to {out_dir}")

In [ ]:
# ── Cell 10: (Optional) Stitch all chapters into one file ─────────
from pathlib import Path

book_name = Path(TXT_DIR).name              # e.g. "chapters" or "sherlock_chapters"
full_mp3  = Path(FULL_OUTPUT_DIR) / f"{book_name}_full.mp3"
mp3s      = sorted(Path(OUTPUT_DIR).glob("*.mp3"))

if not mp3s:
    print(f"No MP3s found in {OUTPUT_DIR} — run Cell 9 first.")
else:
    print(f"Stitching {len(mp3s)} chapter(s)...")
    ffmpeg_concat(mp3s, full_mp3)
    size_mb = full_mp3.stat().st_size / (1024 * 1024)
    print(f"Full audiobook : {full_mp3}  ({size_mb:.1f} MB)")

In [ ]:
# ── Cell 12: Transcribe a chapter MP3 with word-level timestamps ──
#
# By default this transcribes the FIRST chapter in OUTPUT_DIR.
# To pick a specific chapter, set CHECK_MP3 manually:
#   CHECK_MP3 = f"{OUTPUT_DIR}/03_A_Scandal_In_Bohemia.mp3"
# Otherwise leave CHECK_MP3 = None and it auto-selects the first file.

CHECK_MP3 = None    # ← set to a path string to pick a specific chapter, or leave None

# ── Auto-detect if not set ────────────────────────────────────────
available = sorted(Path(OUTPUT_DIR).glob("*.mp3"))
if not available:
    raise FileNotFoundError(f"No MP3s found in {OUTPUT_DIR} — run Cell 9 first.")

if CHECK_MP3 is None:
    CHECK_MP3 = str(available[0])
    print(f"Auto-selected: {Path(CHECK_MP3).name}")

# ── List all available chapters ───────────────────────────────────
print("\nAvailable chapter MP3s:")
for f in available:
    marker = "  ◀ checking this" if f == Path(CHECK_MP3) else ""
    print(f"  {f.name}{marker}")
print()

import whisperx, torch

device       = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"Loading Whisper on {device.upper()}...")
wx_model = whisperx.load_model("base", device, compute_type=compute_type, language="en")

print(f"Transcribing: {Path(CHECK_MP3).name}")
audio  = whisperx.load_audio(CHECK_MP3)
result = wx_model.transcribe(audio, language="en")

print("Aligning word timestamps...")
align_model, metadata = whisperx.load_align_model(language_code="en", device=device)
result = whisperx.align(result["segments"], align_model, metadata, audio, device)

# ── Save timestamped transcript to Drive ─────────────────────────
transcript_path = Path(CHECK_MP3).with_suffix(".transcript.txt")
lines = []
for seg in result["segments"]:
    lines.append(f"\n[{seg['start']:.1f}s — {seg['end']:.1f}s]  {seg['text'].strip()}")
    for w in seg.get("words", []):
        start = w.get("start", "?")
        lines.append(f"    {start if start == '?' else f'{start:.2f}s':>8}  {w['word']}")

transcript_path.write_text("\n".join(lines), encoding="utf-8")
print(f"\nTranscript saved: {transcript_path}")
print("\n--- Preview (first 30 lines) ---")
print("\n".join(lines[:30]))

In [ ]:
# ── Cell 11: Install WhisperX (run once per session) ─────────────
%pip install -q whisperx